In [1]:
"""
-Arman Bazarchi-
Exploitation Zone — Metadata notebook

 - Read trusted metadata CSV(s) from trusted-zone
 - merges values of needed columns of a row into a single string for embedding
 - we keep up to kingdom because model can be used to hold and integrate data of different kingdoms
 - (Animalia, Plante, Fungi)
 - Create text embeddings for text metadata using ChromaDB
 - Avoid duplicates by checking existing uuids in the collection
 - Store embeddings persistently for similarity search
 - we store them in a chroma directory 'exploitation_db' in the 'metadata_embeddings' collection.
"""


from minio import Minio
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
import os, io
from datetime import datetime

# -----------------------
#      Functions
# -----------------------
def get_minio_config():
    # Load MinIO configuration from environment variables (set by orchestrator).
    
    import os
    
    # Get configuration from environment variables (set by orchestrator)
    endpoint = os.getenv('MINIO_ENDPOINT', 'localhost:9000')
    access_key = os.getenv('MINIO_ACCESS_KEY', 'admin')
    secret_key = os.getenv('MINIO_SECRET_KEY', 'admin123')
    
    print(f"Using MinIO configuration from environment variables: endpoint={endpoint}, access_key={access_key[:3]}***")
    return endpoint, access_key, secret_key

# -----------------------
#      Helper Functions
# -----------------------

def _get_script_directory():
    """Get the script directory, handling both orchestrated and notebook execution."""
    try:
        return os.path.dirname(os.path.abspath(__file__))
    except NameError:
        return os.getcwd()

def _setup_minio_client_and_validate(minio, access_key, secret_key, trusted_zone):
    """Setup MinIO client and validate trusted zone bucket exists."""
    client = Minio(minio, access_key=access_key, secret_key=secret_key, secure=False)
    
    if not client.bucket_exists(trusted_zone):
        raise SystemExit(f" Trusted zone bucket '{trusted_zone}' does not exist. Cannot continue.")
    
    return client

def _load_latest_metadata(client, trusted_zone, trusted_meta_prefix):
    """Load the latest metadata CSV from trusted zone."""
    metadata_objs = [
        obj.object_name for obj in client.list_objects(trusted_zone, prefix=trusted_meta_prefix, recursive=True)
        if obj.object_name.lower().endswith(".csv")]
    
    if not metadata_objs:
        raise SystemExit(" No trusted metadata files found in trusted-zone.")
    
    metadata_objs.sort(reverse=True)
    latest_meta = metadata_objs[0]
    print(f" Loading trusted metadata: {latest_meta}")
    
    resp = client.get_object(trusted_zone, latest_meta)
    data = resp.read()
    resp.close()
    resp.release_conn()
    metadata_df = pd.read_csv(io.BytesIO(data))
    print(f" Loaded trusted metadata with {len(metadata_df)} rows.")
    
    return metadata_df

def _prepare_metadata_for_embedding(metadata_df):
    """Combine text columns and clean metadata for embedding."""
    if len(metadata_df) == 0:
        print(" No metadata found in trusted zone. Skipping embedding process.")
        return None
    
    text_cols = ["kingdom", "phylum", "class", "order", "family", "genus", "species", "scientific_name", "common"]
    metadata_df["combined_text"] = metadata_df[text_cols].fillna("").agg(" ".join, axis=1).str.strip()
    
    metadata_df = metadata_df.dropna(subset=["uuid", "combined_text"])
    metadata_df = metadata_df[metadata_df["combined_text"].str.len() > 0]
    print(f" Cleaned metadata for embedding: {len(metadata_df)} valid rows.")
    
    return metadata_df

def _get_existing_uuids(collection):
    """Get all existing UUIDs from the collection."""
    existing_count = collection.count()
    existing_ids = []
    
    if existing_count > 0:
        batch_size = 500
        offset = 0
        while True:
            batch = collection.get(limit=batch_size, offset=offset)
            if not batch["ids"]:
                break
            existing_ids.extend(batch["ids"])
            offset += batch_size
    
    existing_ids = set(existing_ids)
    print(f" Existing embeddings in collection: {len(existing_ids)}")
    return existing_ids, existing_count

def _add_embeddings_in_batches(collection, new_df, texts, uuids, collection_name, text_embedder):
    """Add embeddings to collection in batches with error handling."""
    print(" Generating embeddings...")
    embeddings = text_embedder(texts)
    
    batch_size = 5000
    total = len(uuids)
    start_idx = 0
    
    print(f" Adding {total} new embeddings to collection '{collection_name}' in batches...")
    
    while start_idx < total:
        end_idx = min(start_idx + batch_size, total)
        
        try:
            collection.add(
                ids=uuids[start_idx:end_idx],
                embeddings=embeddings[start_idx:end_idx],
                metadatas=new_df.iloc[start_idx:end_idx].to_dict(orient="records"),
                documents=texts[start_idx:end_idx],
            )
            print(f"  Added records {start_idx + 1}-{end_idx} / {total}")
            start_idx = end_idx
        
        except Exception as e:
            print(f"   Batch {start_idx + 1}-{end_idx} failed: {e}")
            
            if "max batch size" in str(e).lower() and batch_size > 1000:
                old_batch = batch_size
                batch_size = batch_size // 2
                print(f"  Reducing batch size from {old_batch} -> {batch_size} and retrying...")
            else:
                raise
    
    print(f"  Added {total} new embeddings to collection '{collection_name}'.")

def _display_summary(collection_name, existing_count, final_count):
    """Display processing summary."""
    added_count = final_count - existing_count
    
    print("\n ===== Summary =====")
    print(f" Collection: {collection_name}")
    print(f" Previously had: {existing_count}")
    print(f" New added: {added_count}")
    print(f" Total now: {final_count}")
    print("=======================")
    print(" Exploitation Metadata processing complete.")

def _setup_chromadb(chroma_db, collection_name):
    """Setup ChromaDB client, collection, and embedder."""
    chroma_client = chromadb.PersistentClient(path=chroma_db)
    collection = chroma_client.get_or_create_collection(
        name=collection_name,
        metadata={"description": "Embeddings for trusted metadata records"}
    )
    text_embedder = embedding_functions.DefaultEmbeddingFunction()
    return collection, text_embedder

def _filter_and_process_new_records(collection, metadata_df, collection_name, text_embedder):
    """Filter new records and process embeddings."""
    existing_ids, existing_count = _get_existing_uuids(collection)
    new_df = metadata_df[~metadata_df["uuid"].isin(existing_ids)]
    print(f" New records to embed: {len(new_df)}")
    
    if new_df.empty:
        print(" No new metadata to embed. No data was added.")
        return existing_count
    
    texts = new_df["combined_text"].tolist()
    uuids = new_df["uuid"].tolist()
    _add_embeddings_in_batches(collection, new_df, texts, uuids, collection_name, text_embedder)
    final_count = collection.count()
    return final_count

# -----------------------
#      Configuration
# -----------------------
def process_exploitation_metadata():

    # Get MinIO configuration from environment variables (set by orchestrator)
    minio, access_key, secret_key = get_minio_config()

    trusted_zone = "trusted-zone"
    trusted_meta_prefix = "metadata/"
    collection_name = "metadata_embeddings"
    
    # Setup configuration
    script_dir = _get_script_directory()
    chroma_db = os.path.join(script_dir, "../exploitation_db")
    
    # Setup MinIO and load metadata
    client = _setup_minio_client_and_validate(minio, access_key, secret_key, trusted_zone)
    metadata_df = _load_latest_metadata(client, trusted_zone, trusted_meta_prefix)
    metadata_df = _prepare_metadata_for_embedding(metadata_df)
    
    if metadata_df is None:
        return
    
    # Setup ChromaDB
    collection, text_embedder = _setup_chromadb(chroma_db, collection_name)
    
    # Filter and process new records
    existing_count = _filter_and_process_new_records(collection, metadata_df, collection_name, text_embedder)
    
    # Display summary
    final_count = collection.count()
    _display_summary(collection_name, existing_count, final_count)

process_exploitation_metadata();


 Loading trusted metadata: metadata/trusted_metadata_2025_12_30_23_00_23.csv
 Loaded trusted metadata with 767 rows.
 Cleaned metadata for embedding: 767 valid rows.


C:\Users\Arman\AppData\Local\Temp\ipykernel_12144\1754868641.py:86: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  metadata_df["combined_text"] = metadata_df[text_cols].applymap(lambda x: str(x) if pd.notna(x) else "").agg(" ".join, axis=1).str.strip()


 Existing embeddings in collection: 0
 New records to embed: 767
 Generating embeddings...
 Adding 767 new embeddings to collection 'metadata_embeddings' in batches...
  Added records 1-767 / 767
  Added 767 new embeddings to collection 'metadata_embeddings'.

 ===== Summary =====
 Collection: metadata_embeddings
 Previously had: 0
 New added: 767
 Total now: 767
 Exploitation Metadata processing complete.
